In [9]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [10]:
load_dotenv()

True

In [11]:
model=ChatOpenAI()

In [14]:
class LLMState(TypedDict):
    topic:str
    outline:str
    content:str 
    

In [15]:
def gen_outline(state:LLMState)->LLMState:
    topic = state['topic']
    prompt = f"Generate a detailed outline for a blog post about {topic}"
    outline = model.invoke(prompt).content
    state['outline']=outline
    return state

def gen_blog(state:LLMState)->LLMState:
    topic = state['topic']
    outline = state['outline']
    prompt = f"Write a blog post about {topic} based on the following outline: {outline}"
    blog_post = model.invoke(prompt).content
    state['blog_post']=blog_post
    return state

In [16]:
graph =StateGraph(LLMState)

graph.add_node('gen_outline',gen_outline)
graph.add_node('gen_blog',gen_blog)

graph.add_edge(START,'gen_outline')
graph.add_edge('gen_outline','gen_blog')
graph.add_edge('gen_blog',END)

workflow = graph.compile()

In [18]:
initial_state={'topic': 'The benefits of using LangGraph for workflow management'}
final_state = workflow.invoke(initial_state)
print(final_state)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}